# Occupancy Data — Feature Engineering

Takes the raw occupancy CSV and builds a clean feature table with:
- Core occupancy columns (date, occupancy %, day info, season)
- Hungarian public holidays and special dates
- Daily temperature from Open-Meteo (Budapest)

In [1]:
import pandas as pd
import requests
from datetime import date, timedelta
from dateutil.easter import easter
from pathlib import Path




occ = pd.read_csv("../data/raw/occupancy_features.csv", parse_dates=["Date"])
print(f"{len(occ):,} rows  |  {occ['Date'].min().date()} → {occ['Date'].max().date()}")
occ.head()

1,008 rows  |  2023-07-10 → 2026-04-12


,Date,Total_Occ,Occ_Pct,Arr_Rooms,Dep_Rooms,Deduct_Group,Deduct_Indiv,Adl_Chl,day_of_week,day_name,month,month_name,year,is_weekend,season,group_ratio,arrival_pressure,guests_per_room
0,2023-07-10,34,22.52,35,1,0,34,38,0,Monday,7,July,2023,0,Summer,0.0,34,1.117647
1,2023-07-11,37,24.50,11,8,0,37,45,1,Tuesday,7,July,2023,0,Summer,0.0,3,1.216216
2,2023-07-12,39,25.83,20,18,0,39,55,2,Wednesday,7,July,2023,0,Summer,0.0,2,1.410256
3,2023-07-13,35,23.18,6,10,0,35,48,3,Thursday,7,July,2023,0,Summer,0.0,-4,1.371429
4,2023-07-14,23,15.23,9,21,0,23,32,4,Friday,7,July,2023,0,Summer,0.0,-12,1.391304


## 1. Keep only the columns we need

In [2]:
df = occ[["Date", "Occ_Pct", "day_of_week", "day_name", "is_weekend", "season"]].copy()
df = df.rename(columns={"Date": "date", "Occ_Pct": "occupancy_%"})
df.head()

,date,occupancy_%,day_of_week,day_name,is_weekend,season
0,2023-07-10,22.52,0,Monday,0,Summer
1,2023-07-11,24.50,1,Tuesday,0,Summer
2,2023-07-12,25.83,2,Wednesday,0,Summer
3,2023-07-13,23.18,3,Thursday,0,Summer
4,2023-07-14,15.23,4,Friday,0,Summer


## 2. Hungarian public holidays & special dates

Covers:
- All official Hungarian public holidays (fixed + moveable)
- Christmas Eve, NYE, Valentine's Day (not public holidays but operationally significant)

In [ ]:
def hungarian_special_dates(year: int) -> dict[date, str]:
    """Return {date: name} for all special dates in a given year."""
    easter_sun = easter(year)

    dates = {
        # --- Fixed public holidays ---
        date(year, 1, 1):   "new_years_day",
        date(year, 3, 15):  "national_day_mar15",
        date(year, 5, 1):   "labour_day",
        date(year, 8, 20):  "st_stephens_day",
        date(year, 10, 23): "national_day_oct23",
        date(year, 11, 1):  "all_saints_day",
        date(year, 12, 25): "christmas_day",
        date(year, 12, 26): "boxing_day",

        # --- Moveable public holidays ---
        easter_sun - timedelta(days=2): "good_friday",
        easter_sun:                     "easter_sunday",
        easter_sun + timedelta(days=1): "easter_monday",
        easter_sun + timedelta(days=49): "whit_sunday",
        easter_sun + timedelta(days=50): "whit_monday",

        # --- Operationally significant (not public holidays) ---
        date(year, 12, 24): "christmas_eve",
        date(year, 12, 31): "new_years_eve",
        date(year, 2, 14):  "valentines_day",
    }
    return dates


years = range(df["date"].dt.year.min(), df["date"].dt.year.max() + 1)
all_specials: dict[date, str] = {}
for year in years:
    all_specials.update(hungarian_special_dates(year))

df["special_day_name"] = df["date"].dt.date.map(all_specials).fillna("")
df["is_special_day"] = (df["special_day_name"] != "").astype(int)

print(f"{df['is_special_day'].sum()} special days flagged across the dataset:")
df[df["is_special_day"] == 1][["date", "day_name", "special_day_name"]]

45 special days flagged across the dataset:


,date,day_name,special_day_name
41,2023-08-20,Sunday,st_stephens_day
105,2023-10-23,Monday,national_day_oct23
114,2023-11-01,Wednesday,all_saints_day
167,2023-12-24,Sunday,christmas_eve
168,2023-12-25,Monday,christmas_day
169,2023-12-26,Tuesday,boxing_day
174,2023-12-31,Sunday,new_years_eve
175,2024-01-01,Monday,new_years_day
219,2024-02-14,Wednesday,valentines_day
249,2024-03-15,Friday,national_day_mar15


## 3. Weather: Daily Temperatures from Open-Meteo

Budapest coordinates: 47.4979°N, 19.0402°E  
Fetches daily max, min, and mean temperature (°C) from the [Open-Meteo Archive API](https://open-meteo.com/en/docs/historical-weather-api).

In [11]:
start_date = df["date"].min().date()
end_date   = df["date"].max().date()

BUDAPEST_LAT = 47.4979
BUDAPEST_LON = 19.0402

# Archive API covers historical data (up to ~5 days ago)
archive_end = min(end_date, date.today() - timedelta(days=5))

params = {
    "latitude": BUDAPEST_LAT,
    "longitude": BUDAPEST_LON,
    "start_date": str(start_date),
    "end_date": str(archive_end),
    "daily": "temperature_2m_max,temperature_2m_min,temperature_2m_mean",
    "timezone": "Europe/Budapest",
}

resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=30)
resp.raise_for_status()
data = resp.json()["daily"]

weather = pd.DataFrame({
    "date": pd.to_datetime(data["time"]),
    "temp_max": data["temperature_2m_max"],
    "temp_min": data["temperature_2m_min"],
    "temp_mean": data["temperature_2m_mean"],
})

# If the occupancy data extends into very recent days, fill from forecast API
if end_date > archive_end:
    forecast_start = archive_end + timedelta(days=1)
    params_fc = {
        "latitude": BUDAPEST_LAT,
        "longitude": BUDAPEST_LON,
        "start_date": str(forecast_start),
        "end_date": str(end_date),
        "daily": "temperature_2m_max,temperature_2m_min,temperature_2m_mean",
        "timezone": "Europe/Budapest",
    }
    resp_fc = requests.get("https://api.open-meteo.com/v1/forecast", params=params_fc, timeout=30)
    resp_fc.raise_for_status()
    data_fc = resp_fc.json()["daily"]
    weather_fc = pd.DataFrame({
        "date": pd.to_datetime(data_fc["time"]),
        "temp_max": data_fc["temperature_2m_max"],
        "temp_min": data_fc["temperature_2m_min"],
        "temp_mean": data_fc["temperature_2m_mean"],
    })
    weather = pd.concat([weather, weather_fc], ignore_index=True)
    print(f"Archive: {start_date} → {archive_end}  |  Forecast fill: {forecast_start} → {end_date}")
else:
    print(f"Archive covers full range: {start_date} → {archive_end}")

print(f"{len(weather):,} days of weather data fetched")
weather.describe().round(1)

Archive covers full range: 2023-07-10 → 2026-04-12
1,008 days of weather data fetched


,date,temp_max,temp_min,temp_mean
count,1008,1008.0,1008.0,1008.0
mean,2024-11-24 12:00:00,16.8,7.9,12.2
min,2023-07-10 00:00:00,-5.5,-13.4,-8.4
25%,2024-03-17 18:00:00,8.1,1.4,4.7
50%,2024-11-24 12:00:00,16.2,7.1,11.2
75%,2025-08-03 06:00:00,25.3,14.8,19.9
max,2026-04-12 00:00:00,38.2,24.4,30.6
std,NaN,10.0,7.8,8.9


## 4. Merge weather into occupancy

In [12]:
df = df.drop(columns=[c for c in df.columns if c.startswith("temp_")], errors="ignore")
df = df.merge(weather, on="date", how="left")

missing_weather = df["temp_mean"].isna().sum()
if missing_weather:
    print(f"⚠ {missing_weather} days without weather data")
else:
    print("All days have weather data")

print(f"Final shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)

All days have weather data
Final shape: (1008, 11)
Columns: ['date', 'occupancy_%', 'day_of_week', 'day_name', 'is_weekend', 'season', 'special_day_name', 'is_special_day', 'temp_max', 'temp_min', 'temp_mean']


,date,occupancy_%,day_of_week,day_name,is_weekend,season,special_day_name,is_special_day,temp_max,temp_min,temp_mean
0,2023-07-10,22.52,0,Monday,0,Summer,,0,33.8,18.5,26.9
1,2023-07-11,24.50,1,Tuesday,0,Summer,,0,31.0,19.8,26.0
2,2023-07-12,25.83,2,Wednesday,0,Summer,,0,32.3,19.4,25.4
3,2023-07-13,23.18,3,Thursday,0,Summer,,0,29.3,19.8,24.3
4,2023-07-14,15.23,4,Friday,0,Summer,,0,27.5,18.9,23.7
5,2023-07-15,14.57,5,Saturday,1,Summer,,0,31.5,18.0,25.4
6,2023-07-16,16.56,6,Sunday,1,Summer,,0,33.1,19.2,27.0
7,2023-07-17,16.56,0,Monday,0,Summer,,0,34.5,22.3,28.0
8,2023-07-18,17.22,1,Tuesday,0,Summer,,0,30.8,21.4,26.8
9,2023-07-19,20.53,2,Wednesday,0,Summer,,0,31.3,20.8,26.4


In [6]:
df.describe().round(2)

,date,occupancy_%,day_of_week,is_weekend,is_special_day,temp_max,temp_min,temp_mean
count,1008,1008.00,1008.0,1008.00,1008.00,1008.00,1008.00,1008.00
mean,2024-11-24 12:00:00,52.23,3.0,0.29,0.04,16.81,7.91,12.18
min,2023-07-10 00:00:00,4.64,0.0,0.00,0.00,-5.50,-13.40,-8.40
25%,2024-03-17 18:00:00,31.79,1.0,0.00,0.00,8.10,1.38,4.70
50%,2024-11-24 12:00:00,53.64,3.0,0.00,0.00,16.15,7.10,11.25
75%,2025-08-03 06:00:00,71.52,5.0,1.00,0.00,25.32,14.80,19.92
max,2026-04-12 00:00:00,98.68,6.0,1.00,1.00,38.20,24.40,30.60
std,NaN,24.14,2.0,0.45,0.21,10.00,7.84,8.93


## 5. Save

In [14]:
out_path = "../data_prep/outputs/occupancy_enriched.csv"
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}  ({len(df):,} rows)")

Saved → ../data_prep/outputs/occupancy_enriched.csv  (1,008 rows)
